In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/Thesis Data/bestinceptionmodel')


In [ ]:
!pip install streamlit pyngrok


In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from pyngrok import ngrok
from PIL import Image

model = load_model('/content/drive/MyDrive/Thesis Data/inception_finetuned.h5')
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
last_conv_layer_name = 'mixed10'

def load_image(image_file):
    img = Image.open(image_file)
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0

    if len(img_array.shape) == 2:  # Grayscale image
        img_array = np.stack([img_array] * 3, axis=-1)

    img_array = np.expand_dims(img_array, axis=0)
    return img_array

# Grad-CAM logic
def generate_gradcam(image):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    image = tf.cast(image, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image)
        conv_outputs, predictions = grad_model(image)
        loss = predictions[:, tf.argmax(predictions[0])]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = np.maximum(heatmap, 0)
    heatmap /= tf.math.reduce_max(heatmap) + tf.keras.backend.epsilon()
    return heatmap

st.title('**Brain Tumor Classifier** - **Medical Support XAI Tool**')

logo_path = '/content/banner2.webp'
logo = Image.open(logo_path)
st.image(logo, use_container_width=True)

st.markdown("""
> **Disclaimer**: This tool is intended to **complement**, not replace, the expertise of medical professionals. Always consult a healthcare provider for an accurate diagnosis.
""")

st.write("""
### **How to Use:**
1.  **Upload** an MRI scan to detect tumor type and view heatmaps instantly.
2.  **Analyze** the detected tumor with **confidence scores**.
3.  **Examine** the Grad-CAM Heatmap highlighting tumor regions.
4.  **Download** results for further analysis.
""")


image_file = st.file_uploader("**Upload an MRI Image**", type=["jpg", "png", "jpeg"])

if image_file is not None:
    img_array = load_image(image_file)
    detect_clicked = st.button("**Detect Tumor Type**")

    if detect_clicked:
        st.write("**Processing...**")


        pred = model.predict(img_array)
        predicted_class = np.argmax(pred)
        confidence = np.max(pred)
        predicted_label = class_names[predicted_class]


        st.markdown(
            f"<h3 style='color:#D1F6FF;'> Predicted Tumor Type: <b>{predicted_label.upper()}</b></h3>",
            unsafe_allow_html=True
        )
        st.markdown(f"**Confidence:** {confidence*100:.2f}%")

        gradcam_result = generate_gradcam(img_array)
        original_img = np.array(Image.open(image_file))
        original_img = cv2.resize(original_img, (224, 224))
        original_img = cv2.cvtColor(original_img, cv2.COLOR_RGB2BGR)

        heatmap_resized = cv2.resize(gradcam_result.numpy(), (224, 224))
        heatmap_resized = np.uint8(255 * heatmap_resized)
        heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

        if len(original_img.shape) == 2:
            original_img = cv2.cvtColor(original_img, cv2.COLOR_GRAY2BGR)

        superimposed_img = cv2.addWeighted(original_img, 0.6, heatmap_color, 0.4, 0)
        superimposed_rgb = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)

        _, buffer = cv2.imencode('.png', superimposed_rgb)
        downloadable_image = buffer.tobytes()

        col1, col2 = st.columns(2)
        with col1:
            st.markdown("<h4 style='text-align: center;'>Uploaded MRI Image</h4>", unsafe_allow_html=True)
            st.image(img_array[0], use_container_width=True)
        with col2:
            st.markdown("<h4 style='text-align: center;'>Grad-CAM Visualization</h4>", unsafe_allow_html=True)
            st.image(superimposed_rgb, use_container_width=True)
            st.download_button(
                label="Download Grad-CAM Image",
                data=downloadable_image,
                file_name="gradcam_result.png",
                mime="image/png"
            )


        st.write("### Prediction Probabilities")
        for i, cls in enumerate(class_names):
            prob = pred[0][i] * 100
            if i == predicted_class:
                st.markdown(
                    f"<p style='font-size:18px; font-weight:bold; color:#D1F6FF;'>{cls.capitalize()}: {prob:.2f}%</p>",
                    unsafe_allow_html=True
                )
            else:
                st.write(f"{cls.capitalize()}: {prob:.2f}%")


        st.write("### Probability Distribution")
        probs = pred[0] * 100
        colors = ['#A2C5AC' if i != predicted_class else '#2E8B57' for i in range(len(class_names))]

        fig, ax = plt.subplots(figsize=(6, 3))
        bars = ax.barh(class_names, probs, color=colors)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Probability (%)')
        ax.set_title('Prediction Confidence by Class')
        ax.invert_yaxis()

        for i, bar in enumerate(bars):
            ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
                    f'{probs[i]:.2f}%', va='center')

        st.pyplot(fig)

    else:
        st.markdown("<h4 style='text-align: center;'>Uploaded MRI Image</h4>", unsafe_allow_html=True)
        st.image(img_array[0], use_container_width=True)


In [ ]:
!ngrok authtoken 2uuBl4uLbcairkqL6wWcFTPuz3W_2cJkMvJEYR9uMhshGGYYs


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("Streamlit app is live at:", public_url)

!streamlit run app.py &
